# E6: fastText + Linear SVM

**Câu hỏi:** Với cùng vector fastText như E5, Linear SVM có tốt hơn GaussianNB và BoW + SVM (E2) không?

**Quy trình:** `message_clean` → tách từ VnCoreNLP → vector fastText 200 chiều → `LinearSVC` → ngưỡng chọn trên Validation.

**Đầu vào:** chỉ bình luận (`message_clean`). Không dùng `message_normalized`, vì cột này do AI gán nhãn tạo ra và gần như lộ nhãn.

**Gộp từ thư mục `E5+E6`:** `src/nlp.py`, `src/models.py`, `src/evaluation.py`, `step1_segment_and_vectorize.py`, `train_test_e6.py`. Tham số fastText và lưới tham số mô hình giữ nguyên theo code gốc.

**Khác code gốc:**
- **Sửa lỗi emoji:** VnCoreNLP làm hỏng emoji (🤡 thành 嵐), nên chỉ tách phần chữ và giữ nguyên emoji.
- **Tái lập được:** fastText chạy 1 luồng nên ra cùng vector mỗi lần; code gốc chạy nhiều luồng nên mỗi lần một khác.
- **PR-AUC:** dùng `average_precision_score` như E1, E2 để so sánh được.
- **Không lưu tách từ qua Excel:** dùng chung bản tách từ với E1, E2.

**Lưu ý khi đọc kết quả:** 3 file được chia theo từng bình luận, tỷ lệ toxic khoảng 21,7% và nhãn theo Keyword-Priority v2.2.

## Chạy trên Google Colab
1. Vào [colab.research.google.com](https://colab.research.google.com), chọn **Tải lên** và mở notebook này.
2. Chọn **Thời gian chạy → Chạy tất cả**. Không cần GPU.
3. Khi có nút **Chọn tệp**, chọn cùng lúc `train.xlsx`, `val.xlsx`, `test.xlsx`. Muốn so sánh với E5, chọn thêm `metrics.json` trong file zip của E5.
4. Cuối notebook sẽ tải về file zip chứa mô hình, bảng điểm và biểu đồ.

Dữ liệu có nội dung bình luận và tên thật, nên chỉ tải lên tài khoản của bạn.

## 1. Cài đặt

In [ ]:
%pip install -q scikit-learn pandas openpyxl matplotlib joblib py_vncorenlp

import importlib
import importlib.util
import os
import shutil
import sys
from pathlib import Path

# fasttext-wheel có sẵn bản dựng nên dễ cài hơn; nếu không có bản phù hợp thì cài bản gốc
if importlib.util.find_spec("fasttext") is None:
    %pip install -q fasttext-wheel
    importlib.invalidate_caches()
if importlib.util.find_spec("fasttext") is None:
    %pip install -q fasttext
    importlib.invalidate_caches()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # VnCoreNLP cần Java và biến JAVA_HOME để tách từ
    if shutil.which("java") is None:
        !apt-get -qq install -y default-jre-headless
    os.environ["JAVA_HOME"] = str(Path(os.path.realpath(shutil.which("java"))).parents[1])
    print("JAVA_HOME =", os.environ["JAVA_HOME"])

## 2. Cấu hình

In [ ]:
import hashlib
import json
import re
import tempfile
import unicodedata
import urllib.request

import fasttext
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from sklearn.svm import LinearSVC

EXPERIMENT = "E6"
SEED = 42
RUN_TEST = True  # Chỉ bật sau khi đã chốt cấu hình bằng Validation; không chỉnh tham số theo Test

# Tham số fastText giữ nguyên từ code gốc (src/nlp.py, src/data_io.py)
# minCount=5 là mặc định của fastText mà code gốc đang dùng; thread=1 để chạy lại ra đúng cùng vector
# (fastText dùng seed nội bộ cố định, bản fasttext-wheel không nhận tham số seed)
FASTTEXT = dict(model="skipgram", dim=100, epoch=25, lr=0.05, minn=2, maxn=5, minCount=5, thread=1)
EMPTY_TOKENS = {"NO_PARENT", "PARENT_MISSING", "POST_MISSING"}

BASE = Path.cwd().resolve()
CACHE = BASE / "data_splits_chuan"  # Dùng chung bản tách từ với E1, E2
VNCORE_DIR = BASE / "vncorenlp"
OUT = BASE / "results_chuan" / EXPERIMENT

for folder in [CACHE, VNCORE_DIR, OUT]:
    folder.mkdir(parents=True, exist_ok=True)

## 3. Tải 3 file Excel lên

In [ ]:
UPLOAD_AGAIN = False  # Đặt True để thay bằng 3 file khác khi đã có file trong phiên làm việc
REQUIRED = ["train", "val", "test"]


def target_for(name):
    """Colab đổi tên file trùng thành "train (1).xlsx", nên nhận file theo từ khóa rồi lưu về tên chuẩn."""
    low = Path(name).name.lower()
    if low.endswith(".xlsx"):
        found = [key for key in REQUIRED if key in low]
        return BASE / f"{found[0]}.xlsx" if len(found) == 1 else None
    if "metrics" in low and low.endswith(".json"):
        target = BASE / "results_chuan" / "E5" / "metrics.json"  # Mỗi notebook Colab là một máy riêng
        target.parent.mkdir(parents=True, exist_ok=True)
        return target
    return None


def missing_files():
    return [f"{key}.xlsx" for key in REQUIRED if not (BASE / f"{key}.xlsx").exists()]


if IN_COLAB:
    from google.colab import files

    while UPLOAD_AGAIN or missing_files():
        print("Chọn các file Excel. Tên file cần chứa train, val hoặc test. Có thể chọn thêm metrics.json của E5. Còn thiếu:", missing_files())
        uploaded = files.upload()
        assert uploaded, "Chưa chọn file nào. Chạy lại cell này để tải lên."
        for name, content in uploaded.items():
            target = target_for(name)
            if BASE / name != target:
                (BASE / name).unlink(missing_ok=True)
            if target is None:
                print("Bỏ qua:", name)
            else:
                target.write_bytes(content)
                print("Đã nhận", name, "->", target.name)
        UPLOAD_AGAIN = False

assert not missing_files(), f"Thiếu {missing_files()} trong {BASE}. Hãy đặt các file này cạnh notebook."
print("Đủ dữ liệu:", [f"{key}.xlsx" for key in REQUIRED])

## 4. Đọc 3 file dữ liệu

In [ ]:
COLUMNS = ["sample_id", "post_id", "comment_id", "comment_type", "post_message", "parent_message", "message_clean", "label"]
frames = {
    name: pd.read_excel(BASE / f"{name}.xlsx", dtype={"sample_id": str, "post_id": str, "comment_id": str})[COLUMNS]
    for name in ["train", "val", "test"]
}
for frame in frames.values():
    frame["message_clean"] = frame["message_clean"].fillna("")
    assert set(frame["label"]) == {0, 1}

ids = {name: set(frame["sample_id"]) for name, frame in frames.items()}
assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"]), "Trùng sample_id giữa các tập"

train_posts = set(frames["train"]["post_id"])
summary = pd.DataFrame({
    name: {
        "binh_luan": len(frame), "bai_dang": frame["post_id"].nunique(), "toxic": frame["label"].sum(),
        "ty_le_toxic": round(frame["label"].mean(), 4),
        "dong_co_bai_trong_train": frame["post_id"].isin(train_posts).sum() if name != "train" else 0,
    } for name, frame in frames.items()
}).T
summary

## 5. Tách từ bằng VnCoreNLP

In [ ]:
SEG_FILE = CACHE / "message_segmented.csv"
everything = pd.concat(frames.values(), ignore_index=True)


def load_segmenter():
    """py_vncorenlp.download_model cần wget nên lỗi trên Windows; chỉ tải đúng 3 file mà wseg cần."""
    import py_vncorenlp

    root = "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/"
    for name in ["VnCoreNLP-1.2.jar", "models/wordsegmenter/vi-vocab", "models/wordsegmenter/wordsegmenter.rdr"]:
        path = VNCORE_DIR / name
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(root + name, path)
    # VnCoreNLP đổi thư mục làm việc khi khởi tạo, phải trả lại để các đường dẫn sau không sai
    model = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORE_DIR))
    os.chdir(BASE)
    return model


def segment_text(segmenter, text):
    """VnCoreNLP làm hỏng ký tự ngoài BMP (🤣 thành 藍), nên chỉ tách phần chữ và giữ nguyên emoji."""
    pieces = []
    for part in re.split(r"([\U00010000-\U0010FFFF]+)", text):
        if re.fullmatch(r"[\U00010000-\U0010FFFF]+", part):
            pieces.append(part)
        elif part.strip():
            pieces.extend(segmenter.word_segment(part))
    return " ".join(pieces)


cached = pd.read_csv(SEG_FILE, dtype=str, keep_default_na=False) if SEG_FILE.exists() else None
usable = cached is not None and dict(zip(cached["sample_id"], cached["message_clean"])) == dict(zip(everything["sample_id"], everything["message_clean"]))
if not usable:
    segmenter = load_segmenter()
    cached = pd.DataFrame({
        "sample_id": everything["sample_id"], "message_clean": everything["message_clean"],
        "message_segmented": [segment_text(segmenter, t) for t in everything["message_clean"]],
    })
    cached.to_csv(SEG_FILE, index=False, encoding="utf-8")


# Tự kiểm tra: tách từ chỉ thêm khoảng trắng và "_"; VnCoreNLP có đổi vị trí dấu thanh nên so sau khi bỏ dấu
def bare(text):
    return re.sub(r"[\s_]", "", "".join(c for c in unicodedata.normalize("NFD", text) if not unicodedata.combining(c)))


assert all(bare(a) == bare(b) for a, b in zip(cached["message_clean"], cached["message_segmented"])), "Tách từ làm mất ký tự"

segmented = cached.set_index("sample_id")["message_segmented"]
train, val, test = (
    frames[name].assign(message_segmented=frames[name]["sample_id"].map(segmented)) for name in ["train", "val", "test"]
)
print("Đã tách từ", len(segmented), "dòng")

## 6. Học fastText và tạo vector câu

In [ ]:
corpus = [t for t in train["message_segmented"].astype(str) if t.strip() and t not in EMPTY_TOKENS]
with tempfile.NamedTemporaryFile("w", encoding="utf-8", delete=False, suffix=".txt") as tmp:
    tmp.write("\n".join(corpus) + "\n")
try:
    # Chỉ học từ Train để không rò rỉ từ Validation/Test; 1 luồng để E5 và E6 có cùng vector
    ft_model = fasttext.train_unsupervised(input=tmp.name, **FASTTEXT)
finally:
    os.remove(tmp.name)


def sentence_vector(text):
    """Ghép trung bình và cực đại [Mean ; Max] để từ độc hại mạnh không bị loãng trong câu dài."""
    tokens = str(text).split()
    if not tokens or text in EMPTY_TOKENS:
        return np.zeros(FASTTEXT["dim"] * 2, dtype=np.float32)
    vectors = [ft_model.get_word_vector(w) for w in tokens]
    return np.hstack([np.mean(vectors, axis=0), np.max(vectors, axis=0)]).astype(np.float32)


X_train, X_val, X_test = (np.array([sentence_vector(t) for t in f["message_segmented"]], dtype=np.float32) for f in (train, val, test))
y_train, y_val, y_test = (f["label"].values.astype(int) for f in (train, val, test))

assert X_train.shape == (len(train), 200) and X_val.shape == (len(val), 200) and X_test.shape == (len(test), 200)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
VECTOR_HASH = hashlib.sha256(X_train.tobytes()).hexdigest()[:16]
print("Từ vựng fastText:", len(ft_model.words), "| Ma trận Train:", X_train.shape, "| Mã kiểm tra vector:", VECTOR_HASH)

## 7. Hàm đánh giá

In [ ]:
def evaluate(y, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "toxic_precision": precision_score(y, pred, zero_division=0),
        "toxic_recall": recall_score(y, pred, zero_division=0),
        "toxic_f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y, pred),
        "pr_auc": average_precision_score(y, scores),  # Cùng cách tính với E1, E2 để so sánh được
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def plot_results(y, scores, threshold, title):
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay.from_predictions(
        y, (scores >= threshold).astype(int), labels=[0, 1],
        display_labels=["Non-toxic", "Toxic"], cmap="Blues", colorbar=False, ax=left,
    )
    left.set_title(f"Ma trận nhầm lẫn: {title}")
    PrecisionRecallDisplay.from_predictions(y, scores, ax=right, plot_chance_level=True, name=EXPERIMENT)
    right.set_title(f"Đường Precision-Recall lớp Toxic: {title}")
    fig.tight_layout()
    fig.savefig(OUT / f"{title.lower()}_charts.png", dpi=150)
    plt.show()


def by_comment_type(frame, scores):
    return pd.DataFrame({
        kind: {"so_dong": int((frame["comment_type"] == kind).sum()),
               **evaluate(frame["label"].values[frame["comment_type"] == kind], scores[frame["comment_type"] == kind], THRESHOLD)}
        for kind in ["top_level", "reply"]
    }).T.round(4)

## 8. Dò tham số và ngưỡng trên Validation

In [ ]:
MODEL_NAME = "fastText + Linear SVM"
C_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
CLASS_WEIGHTS = [None, "balanced", {0: 1.0, 1: 1.2}, {0: 1.0, 1: 1.5}, {0: 1.0, 1: 1.8}, {0: 1.0, 1: 2.0}]


def toxic_scores(model, X):
    return model.decision_function(X)


rows, best = [], {"val_f1": -1.0}
for C in C_VALUES:
    for class_weight in CLASS_WEIGHTS:
        clf = LinearSVC(C=C, class_weight=class_weight, max_iter=3000, random_state=SEED, dual=False).fit(X_train, y_train)
        scores = toxic_scores(clf, X_val)
        thresholds = np.linspace(*np.percentile(scores, [5, 95]), 51)
        f1s = [f1_score(y_val, (scores >= t).astype(int), zero_division=0) for t in thresholds]
        i = int(np.argmax(f1s))  # argmax lấy ngưỡng đầu tiên, giống vòng lặp của code gốc
        rows.append({"C": C, "class_weight": str(class_weight), "threshold": float(thresholds[i]), "val_f1": f1s[i],
                     "val_pr_auc": average_precision_score(y_val, scores)})
        if f1s[i] > best["val_f1"]:
            best = {"model": clf, "params": {"C": C, "class_weight": str(class_weight)}, "threshold": float(thresholds[i]), "val_f1": f1s[i]}

grid = pd.DataFrame(rows).sort_values("val_f1", ascending=False).reset_index(drop=True)
grid.to_csv(OUT / "grid_search.csv", index=False, encoding="utf-8-sig")
print(len(rows), "cấu hình")
grid.head(10).round(4)

## 9. Kết quả với cấu hình đã chốt

In [ ]:
model, THRESHOLD = best["model"], best["threshold"]
print("Cấu hình chốt:", best["params"], "| ngưỡng:", round(THRESHOLD, 4))

val_scores = toxic_scores(model, X_val)
majority_pred = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_val)
metrics = {
    "Train": evaluate(y_train, toxic_scores(model, X_train), THRESHOLD),
    "Validation": evaluate(y_val, val_scores, THRESHOLD),
    "Lớp phổ biến nhất (Validation)": evaluate(y_val, majority_pred.astype(float), 0.5),
}
pd.DataFrame(metrics).T.round(4)

In [ ]:
plot_results(y_val, val_scores, THRESHOLD, "Validation")

## 10. Phân tích lỗi trên Validation

In [ ]:
display(by_comment_type(val, val_scores))

errors = val.assign(score=val_scores, pred=(val_scores >= THRESHOLD).astype(int))
errors = errors[errors["pred"] != errors["label"]]
errors["loai_loi"] = np.where(errors["label"] == 1, "FN: bỏ sót toxic", "FP: báo nhầm toxic")
errors = errors.sort_values("score", key=lambda s: (s - THRESHOLD).abs(), ascending=False)[
    ["sample_id", "post_id", "loai_loi", "score", "post_message", "parent_message", "message_clean"]
]
errors.to_csv(OUT / "val_errors.csv", index=False, encoding="utf-8-sig")
print(errors["loai_loi"].value_counts().to_dict())
errors[["loai_loi", "score", "message_clean"]].head(15)

## 11. Lưu kết quả

In [ ]:
ft_model.save_model(str(OUT / "fasttext_model.bin"))
joblib.dump({"model": model, "threshold": THRESHOLD, "params": best["params"]}, OUT / "model.joblib")
report = {"experiment": EXPERIMENT, "model": MODEL_NAME, "seed": SEED, "fasttext": FASTTEXT, "pooling": "mean_max",
          "params": best["params"], "threshold": THRESHOLD, "vector_hash": VECTOR_HASH, "metrics": metrics}
(OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu vào", OUT)

## 12. Đánh giá cuối trên Test

In [ ]:
if RUN_TEST:
    test_scores = toxic_scores(model, X_test)
    report["metrics"]["Test"] = evaluate(y_test, test_scores, THRESHOLD)
    (OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    test.assign(score=test_scores, pred=(test_scores >= THRESHOLD).astype(int))[
        ["sample_id", "post_id", "comment_type", "label", "pred", "score", "message_clean"]
    ].to_csv(OUT / "test_predictions.csv", index=False, encoding="utf-8-sig")
    plot_results(y_test, test_scores, THRESHOLD, "Test")
    display(pd.DataFrame({"Test": report["metrics"]["Test"]}).T.round(4))
    display(by_comment_type(test, test_scores))
else:
    print("Chưa đánh giá Test. Đặt RUN_TEST = True sau khi đã chốt cấu hình, rồi chạy lại toàn bộ notebook.")

## 13. So sánh các mô hình trên Test

In [ ]:
# Bảng so sánh tự lấy các mô hình đã có kết quả
keys = ["toxic_precision", "toxic_recall", "toxic_f1", "macro_f1", "accuracy", "pr_auc"]
table = {}
for name in ["E1", "E2", "E5"]:
    path = BASE / "results_chuan" / name / "metrics.json"
    if path.exists() and "Test" in (other := json.loads(path.read_text(encoding="utf-8")))["metrics"]:
        table[name] = {k: other["metrics"]["Test"][k] for k in keys}
if RUN_TEST:
    table["E6"] = {k: report["metrics"]["Test"][k] for k in keys}
pd.DataFrame(table).T.round(4) if table else print("Chưa có kết quả Test để so sánh")

## 14. Tải kết quả về máy

In [ ]:
# Máy Colab bị xóa khi hết phiên, nên tải kết quả về máy trước khi đóng
archive = shutil.make_archive(str(BASE / f"ket_qua_{EXPERIMENT}"), "zip", OUT)
if IN_COLAB:
    files.download(archive)
print("Đã đóng gói:", archive)